### RAG Test and Evaluation

In [1]:
import os
from pathlib import Path
os.chdir(path = Path(r"C:\Users\apaks\projects\YT-RAG"))

In [2]:
from src.yt_rag.components.data_loader import DataLoader
from src.yt_rag.components.embedding import EmbeddingManager
from src.yt_rag.components.vectorstore import FaissVectorStore, VectorStoreManager
from src.yt_rag.components.search import RAGSearch
import time

c:\Users\apaks\projects\YT-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# VectorStoreManager().reset()

In [4]:
url = "https://www.youtube.com/watch?v=5t1vTLU7s40"

In [5]:
query = "How does the RTX 50 Series use AI to process images differently than traditional rendering?"

In [6]:
def run_rag_pipeline(url, query):
    rag = RAGSearch(url = url)
    start = time.time()
    relevant_chunks = rag.search(query= query, top_k = 5)
    context = " ".join(relevant_chunks)
    response = rag.generate_response(context=context, query=query)
    timestamps = rag.get_video_timestamps()
    end = time.time()

    runtime_duration = end-start
    return {"answer": response, "relevant_chunks": relevant_chunks,"timestamps": timestamps, "runtime_duration": runtime_duration}

In [7]:
# print(result)
# print(timestamps)
# print(runtime_duration)

In [8]:
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "YouTube-RAG"

In [9]:
from langsmith import Client

# initiate langsmith client
client = Client()

In [10]:
# Define examples
examples = [
    {
        "inputs": {"question": "Why does Yann LeCun believe auto-regressive LLMs are limited for achieving human-level intelligence?"},
        "outputs": {"answer": "LLMs lack four essential characteristics: an understanding of the physical world, persistent memory, the ability to reason, and the ability to plan. They generate text token-by-token without the abstract, deliberate thought processes humans use before speaking."}
    },
    {
        "inputs": {"question": """What is a "Joint-Embedding Predictive Architecture" (JEPA) and how does it differ from generative models?"""},
        "outputs": {"answer": """Unlike generative models that spend resources attempting to reconstruct every detail or pixel of an input, JEPA learns abstract representations of data. It operates in an abstract space to predict future states without needing to generate the original, full-resolution input."""}
    },
    {
        "inputs": {"question": """Why does LeCun argue that open-sourcing AI is essential for democracy?"""},
        "outputs": {"answer": """Relying on proprietary systems controlled by a small number of companies creates a danger of concentrated power over the "information diet" of citizens. Open source enables diverse AI models that respect different cultures, values, and languages globally."""}
    },
    {
        "inputs": {"question": """How does LeCun suggest machines should perform reasoning and planning in the future?"""},
        "outputs": {"answer": """Future systems should perform planning through an optimization process in an abstract representation space. By using gradient-based inference to minimize an energy function, the model can deliberate on an answer before translating it into text."""}
    },
    {
        "inputs": {"question": """Is it possible to create an AI system that is completely unbiased?"""},
        "outputs": {"answer": """No, it is impossible. Bias is in the eye of the beholder, and different people have conflicting ideas about what constitutes bias. The solution is not to eliminate bias, but to ensure diversity of information sources, similar to a free press."""}
    }
]

In [12]:
# create a dataset and examples in Langsmith
dataset_name = "RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples,
)

{'example_ids': ['f4cda954-38ac-46a4-905f-dbc75a4ae1c9',
  '6f34fbe7-9c40-4552-9ec7-9550b3740c4a',
  '3112aa3c-0adc-4f49-955a-169101fdc0a6',
  'ed3430f0-af6f-48d9-ae42-35339655b95f',
  '2192eaf6-ecc3-4633-a686-28f0ce72162a'],
 'count': 5,
 'as_of': '2026-08-04T23:39:01.495129065Z'}

### Evaluators

Correctness:
- Does the application generates the correct answer. 
    - Goal: Measure how similar or correct is the RAG answer relative to the ground truth
    - Mode: Requires a ground through (reference) answer supplied through evaluation dataset
    - Evaluator: Use LLM as judge to asses the correctness

In [13]:
from typing_extensions import Annotated, TypedDict

class CorrectnessGrade(TypedDict):
    explanation : Annotated[str, "Explain the reasoning for the score"]
    correct: Annotated[bool, "True if the answer is true otherwise False"]

# correctness prompt
correctness_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and a STUDENT ANSWER.

Here is the grading criteria to follow:
1) Grade the student answer based ONLY on their factual accuracy relative to the ground through answer.
2) Ensure that the student answer does not contain any conflicting statement.
3) It is OK if the student answer contains more information than the ground answer, as long as it is factually accurate relative to the ground truth answer. 

Correctness: 
A correctness value of True means that the student's answer meets all the criteria.
A correctness value of False means that the student's answer does not meet all the criteria. 

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
""" 


In [14]:
from langchain_openai import ChatOpenAI

grader_llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0).with_structured_output(CorrectnessGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

# evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs:dict) -> bool:
    answers = f"""
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])

    return grade['correct']

Relevance: Response vs input
- Just check the inputs and output without looking at the reference_outputs. It will answer whether the model address the user's question or not

In [15]:
# grade output schema 
class RelevanceGrade(TypedDict):
    explaination: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the answer addresses the question. False if the answer fails to address the question"]

# relavance instructions
relevance_instructions = """
You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset.
"""

In [16]:
relevance_llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0).with_structured_output(RelevanceGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"""
QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['answer']}
    """

    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])

    return grade['relevant']

Groundedness: Response vs retrived docs
- How relevant is the response to the retrieved docs

In [17]:
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "True if the answer does not halucinates from the documents"]
# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


In [18]:
grounded_llm = ChatOpenAI(model = "gpt-4o-mini", temperature =0).with_structured_output(GroundedGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

def groundedness(inputs: dict, outputs: dict) -> bool:
    retrived_text = "\n\n".join(outputs['relevant_chunks'])
    answer = f"FATCS:{retrived_text}\nSTUDENT ANSWER: {outputs['answer']}"

    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])

    return grade['grounded']

Retrieval Relevance: Retrieved docs vs input

In [19]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


In [20]:
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    retrived_text = "\n\n".join(outputs['relevant_chunks'])
    answer = f"FACTS: {retrived_text}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

## Run Evaluation

In [21]:
from langsmith import traceable

# define the function to run the full rag pipeline
@traceable()
def run_rag_pipeline(url, query):
    rag = RAGSearch(url = url)
    start = time.time()
    relevant_chunks = rag.search(query= query, top_k = 5)
    context = " ".join(relevant_chunks)
    response = rag.generate_response(context=context, query=query)
    timestamps = rag.get_video_timestamps()
    end = time.time()

    runtime_duration = end-start
    return {"answer": response, "relevant_chunks": relevant_chunks,"timestamps": timestamps, "runtime_duration": runtime_duration}

In [22]:
url = "https://www.youtube.com/watch?v=5t1vTLU7s40"

In [23]:
def target(inputs:dict) -> dict:
    return run_rag_pipeline(url = url, query=inputs['question'])

experiment_results = client.evaluate(
    target,
    data = dataset_name,
    evaluators = [correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"}
)

View the evaluation results for experiment: 'rag-doc-relevance-49b23bed' at:
https://smith.langchain.com/o/92a32521-1999-4625-877a-20ef1a977765/datasets/50be5fbf-3a2c-4df5-91dc-aafeb589a124/compare?selectedSessions=d7576ef8-b2f6-439e-bfb3-869763da9849




5it [01:02, 12.57s/it]


In [27]:
eval_results = experiment_results.to_pandas()
eval_results

,inputs.question,outputs.answer,outputs.relevant_chunks,outputs.timestamps,outputs.runtime_duration,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,Is it possible to create an AI system that is ...,"Final Answer: No, it is absolutely not possibl...",[- The fundamental criticism that Gemini is ge...,"[(6448.62, 6562.71), (5704.938, 5851.94), (655...",3.775949,None,"No, it is impossible. Bias is in the eye of th...",True,True,True,True,3.861299,2192eaf6-ecc3-4633-a686-28f0ce72162a,019fcf25-e36e-7da2-9230-eee97a52815d
1,Why does LeCun argue that open-sourcing AI is ...,Final Answer: LeCun argues that open-sourcing ...,[because those systems will constitute the rep...,"[(5958.923, 6091.2), (5704.938, 5851.94), (872...",3.919534,None,Relying on proprietary systems controlled by a...,True,True,True,True,3.937690,3112aa3c-0adc-4f49-955a-169101fdc0a6,019fcf26-16e2-7ce0-a4cb-76e639596a3b
2,"What is a ""Joint-Embedding Predictive Architec...",Final Answer: A Joint-Embedding Predictive Arc...,"[Because you're like French, and ami is I gues...","[(1728.111, 1838.94), (1837.29, 1952.04), (523...",4.195482,None,Unlike generative models that spend resources ...,True,True,True,True,4.217263,6f34fbe7-9c40-4552-9ec7-9550b3740c4a,019fcf26-4af4-79d2-bbb1-6c8a6b9e012b
3,How does LeCun suggest machines should perform...,Final Answer: LeCun suggests that machines sho...,[or running a simulation or calling a calculat...,"[(9319.435, 9442.32), (125.82, 258.93), (9440....",4.127856,None,Future systems should perform planning through...,True,True,True,True,4.142805,ed3430f0-af6f-48d9-ae42-35339655b95f,019fcf26-7c5b-7652-a60c-4d2cbe5fbbca
4,Why does Yann LeCun believe auto-regressive LL...,Final Answer: Yann LeCun believes autoregressi...,[and fascinating discussions online as we do i...,"[(125.82, 258.93), (3268.98, 3377.76), (3021.2...",3.187677,None,LLMs lack four essential characteristics: an u...,True,True,True,True,3.197700,f4cda954-38ac-46a4-905f-dbc75a4ae1c9,019fcf26-aed8-72c1-b773-b7c2ba43a046


In [32]:
import pandas as pd
from datetime import datetime

def save_eval_results(obj:pd.DataFrame, dir_path:str = "eval_results"):
    """Save Evaluation results as excel"""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    dir = Path(dir_path)
    dir.mkdir(exist_ok=True)
    file_path = dir / f"eval_{timestamp}.csv"
    obj.to_csv(file_path)

save_eval_results(obj=eval_results)